# Test AgentCore Long-Term Memory

This notebook tests the memory-enabled agent from Module 6.

**Prerequisites:** Module 6 must be completed first (Gateway + Lambdas deployed).

## What You'll Test

1. Create an AgentCore Memory resource
2. Deploy a second agent with `memory_mode="STM_AND_LTM"`
3. Test cross-session memory recall (same actor, different sessions)


In [ ]:
import boto3
import json
import time
import os
import glob
import uuid

REGION = os.environ.get("AWS_REGION", "us-east-1")
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

# AWS clients
iam = boto3.client("iam")
agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)
codebuild_client = boto3.client("codebuild", region_name=REGION)

# Resource names from Module 6
BOOKINGS_TABLE = "workshop-Bookings"
AGENTCORE_ROLE_NAME = "workshop-AgentCoreExecutionRole"
GATEWAY_NAME = "HotelBookingGateway"

# Recover variables from Module 6 deployment
try:
    AGENTCORE_ROLE_ARN = iam.get_role(RoleName=AGENTCORE_ROLE_NAME)["Role"]["Arn"]
    print(f"AgentCore role: {AGENTCORE_ROLE_ARN}")
except Exception as e:
    raise RuntimeError(f"Module 6 role not found. Deploy Module 6 first. Error: {e}")

try:
    gateways = agentcore.list_gateways()["items"]
    gw = next((g for g in gateways if g["name"] == GATEWAY_NAME), None)
    if not gw:
        raise RuntimeError("Gateway not found")
    GATEWAY_ID = gw["gatewayId"]
    gw_info = agentcore.get_gateway(gatewayIdentifier=GATEWAY_ID)
    GATEWAY_URL = gw_info["gatewayUrl"]
    print(f"Gateway: {GATEWAY_ID}")
    print(f"Gateway URL: {GATEWAY_URL}")
except Exception as e:
    raise RuntimeError(f"Module 6 Gateway not found. Deploy Module 6 first. Error: {e}")

print("\n✓ Module 6 resources recovered. Ready to deploy memory-enabled agent.")

In [ ]:
# Step 1: Create AgentCore Memory resource with boto3from bedrock_agentcore_starter_toolkit import Runtimeagentcore_control = boto3.client('bedrock-agentcore-control', region_name=REGION)MEMORY_RUNTIME_NAME = "HotelBookingAgentWithMemory"MEMORY_NAME = "workshop_HotelBookingMemory"# Create Memory resource with user preference strategyprint("Creating AgentCore Memory resource...")try:    memory_response = agentcore_control.create_memory(        name=MEMORY_NAME,        description="Long-term memory for hotel booking agent - stores user preferences",        eventExpiryDuration=30,  # Keep events for 30 days        memoryStrategies=[            {                'userPreferenceMemoryStrategy': {                    'name': 'UserPreferences',                    'description': 'Store user hotel preferences (stars, cities, etc.)',                    'namespaces': ['users/{actorId}/preferences']                }            },            {                'semanticMemoryStrategy': {                    'name': 'UserFacts',                    'description': 'Store factual information about users',                    'namespaces': ['users/{actorId}/facts']                }            }        ]    )    MEMORY_ID = memory_response['memory']['id']    print(f"Created memory: {MEMORY_ID}")except (agentcore_control.exceptions.ConflictException, agentcore_control.exceptions.ValidationException) as e:    # Memory already exists, get it    memories = agentcore_control.list_memories()    memory = next((m for m in memories['memories'] if m['name'] == MEMORY_NAME), None)    if memory:        MEMORY_ID = memory['id']        print(f"Memory already exists: {MEMORY_ID}")    else:        raise# Wait for memory to be ACTIVEprint("Waiting for memory to be ACTIVE...")for _ in range(30):    memory_status = agentcore_control.get_memory(memoryIdentifier=MEMORY_ID)    if memory_status['status'] == 'ACTIVE':        break    time.sleep(5)print(f"Memory status: {memory_status['status']}")print()# Pre-flight cleanup for the memory-enabled agentCB_PROJECT_MEMORY = f"bedrock-agentcore-{MEMORY_RUNTIME_NAME.lower()}-builder"try:    codebuild_client.delete_project(name=CB_PROJECT_MEMORY)    print(f"Deleted previous CodeBuild project: {CB_PROJECT_MEMORY}")except:    passfor cfg in glob.glob(".bedrock_agentcore*.yaml") + glob.glob(os.path.expanduser("~/.bedrock_agentcore*.yaml")):    try:        os.remove(cfg)    except:        passprint("✅ Pre-flight cleanup done for memory agent\n")# Configure the memory-enabled agentagent_runtime_memory = Runtime()agent_runtime_memory.configure(    entrypoint="booking_agent_with_memory.py",    execution_role=AGENTCORE_ROLE_ARN,    auto_create_ecr=True,    requirements_file="agent_requirements.txt",    region=REGION,    agent_name=MEMORY_RUNTIME_NAME,    memory_mode="STM_AND_LTM",  # ← This enables STM + LTM with strategy extraction    deployment_type="container",    non_interactive=True,)print("Launching agent with long-term memory (3-5 minutes)...")result_memory = agent_runtime_memory.launch(    auto_update_on_conflict=True,    env_vars={        "AWS_REGION": REGION,        "BOOKINGS_TABLE": BOOKINGS_TABLE,        "GATEWAY_URL": GATEWAY_URL,        "BEDROCK_AGENTCORE_MEMORY_ID": MEMORY_ID,  # ← Pass memory ID to agent    },)MEMORY_RUNTIME_ARN = result_memory.agent_arnMEMORY_RUNTIME_ID = MEMORY_RUNTIME_ARN.split("/")[-1] if MEMORY_RUNTIME_ARN else Noneprint(f"\n✅ Memory-enabled agent deployed: {MEMORY_RUNTIME_ARN}")print(f"Memory ID: {MEMORY_ID}")print(f"\nThis agent will remember conversations across different session IDs.")

### Test Long-term Memory

Now we'll test that the agent remembers information **across different sessions**.

**Test scenario:**
1. **Session A:** Tell the agent your name and preferred city
2. **Wait 60 seconds** for AgentCore to extract strategies asynchronously
3. **Session B:** Ask about hotels (different session ID, same user ID)
4. **Verify:** The agent should remember your name and preference from Session A

#### How to Set Actor ID for Long-Term Memory

AgentCore uses **actor ID** to identify whose strategies to store and retrieve. The actor ID is passed via a **custom HTTP header**, not an API parameter.

**Implementation with boto3 event system:**

```python
client = boto3.client('bedrock-agentcore', region_name=region)
event_system = client.meta.events

def add_custom_runtime_header(request, **kwargs):
    request.headers.add_header('X-Amzn-Bedrock-AgentCore-Runtime-Custom-Actor-Id', user_id)

handler = event_system.register_first('before-sign.bedrock-agentcore.InvokeAgentRuntime', add_custom_runtime_header)
```

**Key points:**
- Actor ID format: `user-{8-char-uuid}` (e.g., `user-a1b2c3d4`)
- Same actor ID across sessions → Agent recalls strategies
- Different actor ID → Different memory space
- The starter toolkit does NOT support this — must use boto3 directly

In [ ]:
import json
from datetime import datetime, timedelta

def wait_for_memory_propagation(seconds=60):
    """Wait for AgentCore to extract long-term memory strategies."""
    print(f"⏳ Waiting {seconds} seconds for long-term memory extraction...")
    print("Note: Long-term memory is an asynchronous process.")
    
    end_time = datetime.now() + timedelta(seconds=seconds)
    interval = 10
    last_update = datetime.now()
    
    while datetime.now() < end_time:
        remaining = int((end_time - datetime.now()).total_seconds())
        if remaining <= 0:
            break
        
        if (datetime.now() - last_update).total_seconds() >= interval:
            print(f"   ... {remaining} seconds remaining")
            last_update = datetime.now()
        
        pause_until = datetime.now() + timedelta(seconds=min(1, remaining))
        while datetime.now() < pause_until:
            pass
    
    print("✓ Wait completed")

def invoke_agent_memory_boto3(agent_arn, prompt, session_id, user_id, region=REGION):
    """
    Invoke agent with long-term memory using boto3 directly.
    
    Uses boto3 event system to add custom actor ID header.
    This is the ONLY way to properly set actor ID for LTM.
    """
    client = boto3.client('bedrock-agentcore', region_name=region)
    event_system = client.meta.events
    
    EVENT_NAME = 'before-sign.bedrock-agentcore.InvokeAgentRuntime'
    CUSTOM_HEADER_NAME = 'X-Amzn-Bedrock-AgentCore-Runtime-Custom-Actor-Id'
    
    def add_custom_runtime_header(request, **kwargs):
        request.headers.add_header(CUSTOM_HEADER_NAME, user_id)
    
    try:
        handler = event_system.register_first(EVENT_NAME, add_custom_runtime_header)
        
        payload = json.dumps({"prompt": prompt}).encode()
        response = client.invoke_agent_runtime(
            agentRuntimeArn=agent_arn,
            runtimeSessionId=session_id,
            payload=payload,
            qualifier="DEFAULT"
        )
        
        content = []
        for chunk in response.get("response", []):
            content.append(chunk.decode('utf-8'))
        
        result = json.loads(''.join(content))
        
        event_system.unregister(EVENT_NAME, handler)
        
        return result.get('response', 'No response')
    
    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        return None

In [ ]:
# Session A: Store preferences and facts
session_a = str(uuid.uuid4())
user_alex = f"user-{str(uuid.uuid4())[:8]}"  # 8-char user ID

print("=" * 70)
print("SESSION A: Teaching the agent about user preferences")
print("=" * 70)
print(f"\nUser ID:    {user_alex}")
print(f"Session ID: {session_a}")
print()

# Test 1: Tell the agent your name and preferences
prompt_a = "My name is Alex and I prefer 4-star hotels in Paris. I also have a loyalty membership number: HOTEL-12345."
print(f"User: {prompt_a}")
print("-" * 70)

response_a = invoke_agent_memory_boto3(
    agent_arn=MEMORY_RUNTIME_ARN,
    prompt=prompt_a,
    session_id=session_a,
    user_id=user_alex,
    region=REGION
)

print(f"Agent: {response_a}")
print()

# Test 2: Within the same session, verify STM works
prompt_a2 = "What's my loyalty number?"
print(f"User: {prompt_a2}")
print("-" * 70)

response_a2 = invoke_agent_memory_boto3(
    agent_arn=MEMORY_RUNTIME_ARN,
    prompt=prompt_a2,
    session_id=session_a,  # Same session
    user_id=user_alex,
    region=REGION
)

print(f"Agent: {response_a2}")
print()
print("✓ Session A completed. The agent remembered within the session (STM).")
print()

# Wait for memory extraction
print("⏳ Now waiting for AgentCore to extract long-term strategies...")
print("   Memory extraction is an ASYNCHRONOUS background process.")
print("   AgentCore analyzes the conversation and identifies:")
print("   - Facts: name, loyalty number")
print("   - Preferences: 4-star hotels, Paris")
print()
wait_for_memory_propagation(60)


In [ ]:
# Session B: NEW session ID, same user — test cross-session memory recall
session_b = str(uuid.uuid4())

print("=" * 70)
print("SESSION B: Testing cross-session memory recall")
print("=" * 70)
print(f"\nUser ID:    {user_alex}  ← SAME user as Session A")
print(f"Session ID: {session_b}  ← DIFFERENT session")
print()
print("The agent should recall:")
print("  - Fact: Your name is Alex")
print("  - Fact: Loyalty number is HOTEL-12345")
print("  - Preference: You prefer 4-star hotels in Paris")
print()

# Test 1: Ask the agent what it remembers
prompt_b1 = "Do you remember me? What's my name?"
print(f"User: {prompt_b1}")
print("-" * 70)

response_b1 = invoke_agent_memory_boto3(
    agent_arn=MEMORY_RUNTIME_ARN,
    prompt=prompt_b1,
    session_id=session_b,  # Different session
    user_id=user_alex,      # Same user
    region=REGION
)

print(f"Agent: {response_b1}")
print()

# Test 2: Ask about loyalty number (fact)
prompt_b2 = "What's my loyalty number?"
print(f"User: {prompt_b2}")
print("-" * 70)

response_b2 = invoke_agent_memory_boto3(
    agent_arn=MEMORY_RUNTIME_ARN,
    prompt=prompt_b2,
    session_id=session_b,
    user_id=user_alex,
    region=REGION
)

print(f"Agent: {response_b2}")
print()

# Test 3: Ask for hotel recommendation (preference)
prompt_b3 = "Find me a hotel based on my preferences"
print(f"User: {prompt_b3}")
print("-" * 70)

response_b3 = invoke_agent_memory_boto3(
    agent_arn=MEMORY_RUNTIME_ARN,
    prompt=prompt_b3,
    session_id=session_b,
    user_id=user_alex,
    region=REGION
)

print(f"Agent: {response_b3}")
print()
print("✓ Session B completed.")
print()
print("=" * 70)
print("MEMORY TEST RESULTS")
print("=" * 70)
print("✓ STM (Short-term): Agent remembered within Session A")
print("✓ LTM (Long-term): Agent recalled facts and preferences in Session B")
print()
print("Key insight:")
print("  - Same session_id → Agent uses STM (conversation buffer)")
print("  - Different session_id + same actor_id → Agent uses LTM (extracted strategies)")
